### Get gene list from composite scoring results
### Julian Moran
### 2026-09-02

In [1]:
import boto3
import glob
import logging
import math
import os
import requests
import s3fs
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv
from pathlib import Path

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [10]:
# ============================================================
#       Args
# ============================================================

# Notebook variables
COMP_THRESHOLD = 0.65

# File intake
BUCKET = "iei-project"
PREFIX_GOLD = "03_gold/defense_finder/"
PREFIX_SILVER = "02_silver/defense_finder/"
FILE_FINAL_OUTPUT = "human_bacteria_structural_analogs_enriched.parquet"
FILE_IUIS_IEI = f"{REPO_ROOT}/data_sync/IUIS-IEI_geneList_cleaned.tsv"

# Dirs
OUT_DIR_PLOT = f"{REPO_ROOT}/vis/plots"

# Check live objects in MinIO silver
client = boto3.client(
    "s3",
    endpoint_url="http://eagle.tcag.ca:9000",
    aws_access_key_id=MINIO_SECRET,
    aws_secret_access_key=MINIO_KEY,
)
response = client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX_GOLD
)
live_objects = [
    obj["Key"]
    for obj in response.get("Contents", [])
]
live_objects

['03_gold/defense_finder/composite_score.parquet/_SUCCESS',
 '03_gold/defense_finder/composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet',
 '03_gold/defense_finder/conserved_modules.parquet',
 '03_gold/defense_finder/defense_human_domain_annotated.parquet',
 '03_gold/defense_finder/defense_islands.manifest.json',
 '03_gold/defense_finder/defense_islands.parquet',
 '03_gold/defense_finder/final_output.parquet',
 '03_gold/defense_finder/final_output_spark.parquet/_SUCCESS',
 '03_gold/defense_finder/final_output_spark.parquet/part-00000-bb778272-db05-479f-a79a-2d18e89fe363-c000.snappy.parquet',
 '03_gold/defense_finder/griid_gene_subset.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.manifest.json',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.manifest.json',
 '03_gold/defense_finder/human_bacteria_structural_analogs_

In [3]:
# ============================================================
#       In -- `df_comp_score`
# ============================================================

df_comp_score = pl.read_parquet(
    f"s3://{BUCKET}/{PREFIX_GOLD}{FILE_FINAL_OUTPUT}",
    storage_options={
        "aws_endpoint_url": "http://eagle.tcag.ca:9000",
        "aws_access_key_id": MINIO_SECRET,
        "aws_secret_access_key": MINIO_KEY,
    }
).drop([
    "Entry", 
    "Entry Name",
    "Organism",
    "Gene Names (ordered locus)",
    "Gene Names (ORF)",
    "Gene Names (synonym)"
]).rename({
    "Reviewed": "human_uniprot_reviewed",
    "cluster_repId": "defense_foldseek_repId",
    "human_entryId": "human_uniprot_ac",
    "Protein names": "human_protein_names",
    "Gene Names": "human_gene_names",
    "Length": "human_length_AAs",
    "Gene Names (primary)": "human_gene_name_primary",
    "Proteomes": "uniprot_ref_proteome_chr"
}).sort("composite_score", descending=True)

# `df_comp_score: count number of non-null gene names
col = "human_gene_name_primary"
n = df_comp_score.height
non_null = df_comp_score[col].is_not_null().sum()
p_w_gene_name = round(non_null / n, 2)

logger.info(f"Pipeline output with non-null gene names {p_w_gene_name}")

df_comp_score

INFO:__main__:Pipeline output with non-null gene names 0.67


defense_uniprot_ac,defense_foldseek_repId,defense_taxId,human_uniprot_ac,human_uniprot_reviewed,human_protein_names,human_gene_names,human_length_AAs,human_gene_name_primary,uniprot_ref_proteome_chr,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (molecular function),Gene Ontology (GO),Gene Ontology IDs,FunFam,CDD,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,defense_type,defense_subtype,defense_species,defense_sys_id,defense_sys_beg,defense_sys_end,defense_duplicate_count,defense_rank,human_domains,defense_gene_name,tm_score_human,tm_score_bacteria,rmsd,pymol_rmsd,pymol_aligned_atoms,pymol_aligned_residues,human_aligned_start,human_aligned_end,bact_aligned_start,bact_aligned_end,evalue_foldseek,griid_gene,iuis_iei,iuis_diseases,bacterial_length,bacterial_domains,in_conserved_module,module_count,module_partner_genes,plddt_human_mean,plddt_bacteria_mean,tm_pvalue_human,tm_pvalue_bacteria,tm_pvalue,q_value,structure_source_human,structure_source_bacteria,alignment_status,immune_relevant,composite_score,alphafold_model_version,run_date
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,f64,bool,bool,str,i64,str,bool,i64,str,f64,f64,f64,f64,f64,f64,str,str,str,bool,f64,str,str
"""Q47ZY9""","""A0A2E5CU08""","""167879""","""Q8IZH9""","""unreviewed""","""GTP cyclohydrolase 1 (EC 3.5.4…",null,"""177""",null,null,"""regulation of blood pressure […","""cytoplasm [GO:0005737]; nucleu…","""GTP binding [GO:0005525]; GTP …","""cytoplasm [GO:0005737]; nucleu…","""GO:0003934; GO:0005525; GO:000…","""1.10.286.10:FF:000003;3.30.113…","""cd00642;""","""1.10.286.10;3.30.1130.10;""","""IPR043133;IPR043134;IPR001474;…","""PTHR11109:SF11;PTHR11109;""","""PF01227;""","""PS00859;PS00860;""",null,"""NP_001019195.1;""","""Dpd""","""Dpd""","""Colwellia psychrerythraea""","""GCF_000012325_NC_003910_Dpd_1""","""GCF_000012325.1_NC_003910_0264…","""GCF_000012325.1_NC_003910_0265…",1,202,"""DOMAIN 2..176; /note=""GTP cycl…","""Dpd__FolE""",0.991,0.964,0.51,0.28,986,175,1,176,6,181,0.0,false,false,null,182,"""DOMAIN 3..180; /note=""GTP cycl…",false,0,null,95.53,97.12,8.8818e-16,2.5535e-15,2.5535e-15,1.7883e-9,"""afdb_v6""","""afdb_v6""","""ok""",false,0.9962,"""model_v6""",""""""
"""A0A5B9E1W6""","""A0A7C2NS94""","""582702""","""Q5QP19""","""unreviewed""","""Cysteine desulfurase (EC 2.8.1…","""NFS1 hCG_38220""","""397""","""NFS1""",null,"""[2Fe-2S] cluster assembly [GO:…","""cytosol [GO:0005829]; mitochon…","""cysteine desulfurase activity …","""cytosol [GO:0005829]; mitochon…","""GO:0005654; GO:0005829; GO:000…","""3.40.640.10:FF:000003;3.90.115…",null,"""3.90.1150.10;3.40.640.10;""","""IPR000192;IPR020578;IPR010240;…","""PTHR11601:SF34;PTHR11601;""","""PF00266;""","""PS00595;""",null,null,"""Dnd""","""Dnd_ABCDE""","""Rhodanobacter glycinis""","""GCF_008000795_NZ_CP042807_Dnd_…","""GCF_008000795.1_NZ_CP042807_02…","""GCF_008000795.1_NZ_CP042807_02…",1,202,"""DOMAIN 2..361; /note=""Aminotra…","""Dnd__DndA""",0.9237,0.9496,1.3,0.54,1874,367,1,376,9,386,0.0,false,false,null,386,"""DOMAIN 7..369; /note=""Aminotra…",false,0,null,94.52,95.11,1.3656e-14,4.6629e-15,1.3656e-14,1.7883e-9,"""afdb_v6""","""afdb_v6""","""ok""",false,0.9863,"""model_v6""",""""""
"""A0A1D7QWY7""","""A0A7C2NS94""","""632773""","""Q5QP19""","""unreviewed""","""Cysteine desulfurase (EC 2.8.1…","""NFS1 hCG_38220""","""397""","""NFS1""",null,"""[2Fe-2S] cluster assembly [GO:…","""cytosol [GO:0005829]; mitochon…","""cysteine desulfurase activity …","""cytosol [GO:0005829]; mitochon…","""GO:0005654; GO:0005829; GO:000…","""3.40.640.10:FF:000003;3.90.115…",null,"""3.90.1150.10;3.40.640.10;""","""IPR000192;IPR020578;IPR010240;…","""PTHR11601:SF34;PTHR11601;""","""PF00266;""","""PS00595;""",null,null,"""Dnd""","""Dnd_ABCDEFGH""","""Salisediminibacterium beveridg…","""GCF_001721685_NZ_CP012502_Dnd_…","""GCF_001721685.1_NZ_CP012502_02…","""

In [11]:
# ============================================================
#       In -- IUIS-IEI
# ============================================================

df_iuis_iei = pl.read_csv(
    FILE_IUIS_IEI,
    separator="\t",
    has_header=True,
    null_values="NA",
    infer_schema_length=None,
)

df_iuis_iei

Major category,Subcategory,Disease,Genetic defect,Inheritance,score_positive_total,VariantCounts.VRE,VariantCounts.ClnVar,OMIM ID,AlphaFold_URL,Uniprot,HPO IDs,HPO term,ICD9,ICD10,Associated features,B cell,T cell,Ig,Neutrophil,Inheritance detail,Major_category_original,GOF/DN details,T cell details,B cell details,Ig details,Neutrophil details,Other affected cells,score5.ClnVar,score4.ClnVar,score2.ClnVar,score0.ClnVar,score5.VRE,score4.VRE,score2.VRE,score0.VRE,min_prob,median_prob,max_prob,q1_prob,q3_prob
str,str,str,str,str,i64,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64
"""1. CID""","""1. T-B+ SCID""","""CD3z deficiency""","""CD247""","""AR""",2,"""1 / 0 / 33 / 1""","""15 / 0 / 133 / 218""","""186780""","""https://alphafold.ebi.ac.uk/en…","""P20963""","""HP:0002715; HP:0005403""","""Abnormality of the immune syst…",279.2,"""D81.2""",null,"""normal""","""low""","""low""","""normal""",null,"""1. Immunodeficiencies affectin…",null,"""Very low""","""Normal""","""Low""","""Normal""","""No g/d T cells""",15,0,133,218,1,0,33,1,1.7997e-8,2.6799e-7,5.1799e-7,1.4299e-7,3.9299e-7
"""1. CID""","""1. T-B+ SCID""","""CD3d deficiency""","""CD3D""","""AR""",3,"""2 / 1 / 34 / 0""","""20 / 10 / 162 / 234""","""186790""","""https://alphafold.ebi.ac.uk/en…","""P04234""","""HP:0002715; HP:0005403""","""Abnormality of the immune syst…",279.2,"""D81.2""",null,"""normal""","""low""","""low""","""normal""",null,"""1. Immunodeficiencies affectin…",null,"""Very low""","""Normal""","""Low""","""Normal""","""No g/d T cells""",20,10,162,234,2,1,34,0,2.0956e-8,2.0976e-8,1.0403e-7,2.0966e-8,6.2503e-8
"""1. CID""","""1. T-B+ SCID""","""CD3e deficiency""","""CD3E""","""AR""",3,"""1 / 2 / 29 / 2""","""26 / 14 / 173 / 346""","""186830""","""https://alphafold.ebi.ac.uk/en…","""P07766""","""HP:0002715; HP:0005403""","""Abnormality of the immune syst…",279.2,"""D81.2""",null,"""normal""","""low""","""low""","""normal""",null,"""1. Immunodeficiencies affectin…",null,"""Very low""","""Normal""","""Low""","""Normal""","""No g/d T cells""",26,14,173,346,1,2,29,2,1.2018e-7,1.2023e-7,9.5964e-7,1.2020e-7,5.3993e-7
"""1. CID""","""1. T-B+ SCID""","""Coronin-1A deficiency""","""CORO1A""","""AR""",2,"""1 / 1 / 43 / 2""","""19 / 14 / 236 / 376""","""605000""","""https://alphafold.ebi.ac.uk/en…","""P31146""","""HP:0002715; HP:0005403""","""Abnormality of the immune syst…",279.2,"""D81.2""","""Detectable thymus, EBV""","""normal""","""low""","""low""","""normal""",null,"""1. Immunodeficiencies affectin…",null,"""Very low""","""Normal""","""Low""","""Normal""",null,19,14,236,376,1,1,43,2,6.8292e-8,6.8292e-8,6.8292e-8,6.8292e-8,6.8292e-8
"""1. CID""","""1. T-B+ SCID""","""gc deficiency (common gamma ch…","""IL2RG""","""XL""",3,"""1 / 1 / 16 / 28""","""194 / 104 / 244 / 414""","""308380""","""https://alphafold.ebi.ac.uk/en…","""P31785""","""HP:0002715; HP:0005403""","""Abnormality of the immune syst…",279.2,"""D81.2""",null,"""borderline""","""low""","""low""","""normal""",null,"""1. Immunodeficiencies affectin…",null,"""Very low""","""Normal to high""","""Low""","""Normal""","""Low NK""",194,104,244,414,1,1,16,28,4.0313e-8,4.0313e-8,1.6147e-7,4.0313e-8,1.0089e-7
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""9. BMF""",null,"""BMFS5""","""TP53""","""AD""",null,null,"""1406 / 325 / 3028 / 1993""","""618165""","""https://alphafold.ebi.ac.uk/en…","""P04637""","""HP:0002721; HP:0005528""","""Immunodeficiency; Bone marrow …",238.72,"""D61.89""","""Erythroid hypoplasia, B-cell d…","""low""",null,null,null,null,"""9. Bone marrow failure""",null,null,"""Low""",null,null,"""Hematopoietic stem cell""",1406,325,3028,1993,null,null,null,null,0.0,0.0,0.0,0.0,0.0
"""9. BMF""",null,"""Fanconi Anemia Type T""","""UBE2T""","""AR""",null,null,"""21 / 5 / 33 / 54""","""616435""","""https://alphafold.ebi.ac.uk/en…","""Q9NPD8""","""HP:0002721; HP:0005528""","""Immun

In [9]:
# ============================================================
#       Pare down, clean up `df_comp_score`
# ============================================================

# Pare down to ...
#   - only highest-composite-score pair for each unique human UniProt accession

df_comp_unq_hs_proteins = df_comp_score.unique(
    subset=["human_uniprot_ac"],
    keep="first",
    maintain_order=True
)

# Count number of unique human gene names in this dataset
df_comp_unq_hs_genes = df_comp_score.unique(subset=["human_gene_name_primary"], keep="first", maintain_order=True)
logger.info(f"Number of unique genes in this dataset: {len(df_comp_unq_hs_genes):,}")
df_comp_unq_hs_proteins

INFO:__main__:Number of unique genes in this dataset: 3,748


defense_uniprot_ac,defense_foldseek_repId,defense_taxId,human_uniprot_ac,human_uniprot_reviewed,human_protein_names,human_gene_names,human_length_AAs,human_gene_name_primary,uniprot_ref_proteome_chr,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (molecular function),Gene Ontology (GO),Gene Ontology IDs,FunFam,CDD,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,defense_type,defense_subtype,defense_species,defense_sys_id,defense_sys_beg,defense_sys_end,defense_duplicate_count,defense_rank,human_domains,defense_gene_name,tm_score_human,tm_score_bacteria,rmsd,pymol_rmsd,pymol_aligned_atoms,pymol_aligned_residues,human_aligned_start,human_aligned_end,bact_aligned_start,bact_aligned_end,evalue_foldseek,griid_gene,iuis_iei,iuis_diseases,bacterial_length,bacterial_domains,in_conserved_module,module_count,module_partner_genes,plddt_human_mean,plddt_bacteria_mean,tm_pvalue_human,tm_pvalue_bacteria,tm_pvalue,q_value,structure_source_human,structure_source_bacteria,alignment_status,immune_relevant,composite_score,alphafold_model_version,run_date
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,f64,bool,bool,str,i64,str,bool,i64,str,f64,f64,f64,f64,f64,f64,str,str,str,bool,f64,str,str
"""Q47ZY9""","""A0A2E5CU08""","""167879""","""Q8IZH9""","""unreviewed""","""GTP cyclohydrolase 1 (EC 3.5.4…",null,"""177""",null,null,"""regulation of blood pressure […","""cytoplasm [GO:0005737]; nucleu…","""GTP binding [GO:0005525]; GTP …","""cytoplasm [GO:0005737]; nucleu…","""GO:0003934; GO:0005525; GO:000…","""1.10.286.10:FF:000003;3.30.113…","""cd00642;""","""1.10.286.10;3.30.1130.10;""","""IPR043133;IPR043134;IPR001474;…","""PTHR11109:SF11;PTHR11109;""","""PF01227;""","""PS00859;PS00860;""",null,"""NP_001019195.1;""","""Dpd""","""Dpd""","""Colwellia psychrerythraea""","""GCF_000012325_NC_003910_Dpd_1""","""GCF_000012325.1_NC_003910_0264…","""GCF_000012325.1_NC_003910_0265…",1,202,"""DOMAIN 2..176; /note=""GTP cycl…","""Dpd__FolE""",0.991,0.964,0.51,0.28,986,175,1,176,6,181,0.0,false,false,null,182,"""DOMAIN 3..180; /note=""GTP cycl…",false,0,null,95.53,97.12,8.8818e-16,2.5535e-15,2.5535e-15,1.7883e-9,"""afdb_v6""","""afdb_v6""","""ok""",false,0.9962,"""model_v6""",""""""
"""A0A5B9E1W6""","""A0A7C2NS94""","""582702""","""Q5QP19""","""unreviewed""","""Cysteine desulfurase (EC 2.8.1…","""NFS1 hCG_38220""","""397""","""NFS1""",null,"""[2Fe-2S] cluster assembly [GO:…","""cytosol [GO:0005829]; mitochon…","""cysteine desulfurase activity …","""cytosol [GO:0005829]; mitochon…","""GO:0005654; GO:0005829; GO:000…","""3.40.640.10:FF:000003;3.90.115…",null,"""3.90.1150.10;3.40.640.10;""","""IPR000192;IPR020578;IPR010240;…","""PTHR11601:SF34;PTHR11601;""","""PF00266;""","""PS00595;""",null,null,"""Dnd""","""Dnd_ABCDE""","""Rhodanobacter glycinis""","""GCF_008000795_NZ_CP042807_Dnd_…","""GCF_008000795.1_NZ_CP042807_02…","""GCF_008000795.1_NZ_CP042807_02…",1,202,"""DOMAIN 2..361; /note=""Aminotra…","""Dnd__DndA""",0.9237,0.9496,1.3,0.54,1874,367,1,376,9,386,0.0,false,false,null,386,"""DOMAIN 7..369; /note=""Aminotra…",false,0,null,94.52,95.11,1.3656e-14,4.6629e-15,1.3656e-14,1.7883e-9,"""afdb_v6""","""afdb_v6""","""ok""",false,0.9863,"""model_v6""",""""""
"""Q47ZY9""","""A0A2E5CU08""","""167879""","""Q8IZH8""","""unreviewed""","""GTP cyclohydrolase 1 (EC 3.5.4…",null,"""158""",null,null,"""regulation of blood pressure […","""cytoplasm [GO:0005737]; nucleu…","""GTP binding [GO:0005525]; GTP …","""cytoplasm [GO:0005737]; nucleu…","""GO:0003934; GO:0005525; GO:000…","""1.10.286.10:FF:000003;3.30.113…",null,"""1.10.286.10;3.30.1130.10;""","""IPR043133;IPR043134;IPR001474;…","""PTHR11109:SF11;PTHR11109;""","""PF01227;""","""PS00859;PS00860;""",null,null,"""Dpd""","""Dpd""","""Colwellia psychrerythraea""","""GCF_000012325_NC_003910_Dpd_1""","""GCF_000012325.1_NC_003910_0264…","""GCF_000012325.1_NC_003910_0265…

In [22]:
# ============================================================
#       Annotate by IUIS-IEI membership
# ============================================================

iuis_iei_genes = df_iuis_iei["Genetic defect"]
df_comp_ann = df_comp_unq_hs_proteins.with_columns(
    pl.col("human_gene_name_primary")
    .is_in(iuis_iei_genes.implode())
    .alias("human_gene_in_IUIS")
)

# Measure how many rows have a valid IUIS-IEI annotation
n_rows_w_iuis = df_comp_ann["human_gene_in_IUIS"].sum()
logger.info(f"n rows with IUIS-IEI membership: {n_rows_w_iuis}")
p_rows_w_iuis = round((df_comp_ann["human_gene_in_IUIS"].sum() / len(df_comp_ann)), 4)
logger.info(f"p rows with IUIS-IEI membership: {p_rows_w_iuis}")

df_comp_ann

INFO:__main__:n rows with IUIS-IEI membership: 387
INFO:__main__:p rows with IUIS-IEI membership: 0.0249


defense_uniprot_ac,defense_foldseek_repId,defense_taxId,human_uniprot_ac,human_uniprot_reviewed,human_protein_names,human_gene_names,human_length_AAs,human_gene_name_primary,uniprot_ref_proteome_chr,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (molecular function),Gene Ontology (GO),Gene Ontology IDs,FunFam,CDD,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,defense_type,defense_subtype,defense_species,defense_sys_id,defense_sys_beg,defense_sys_end,defense_duplicate_count,defense_rank,human_domains,defense_gene_name,tm_score_human,tm_score_bacteria,rmsd,pymol_rmsd,pymol_aligned_atoms,pymol_aligned_residues,human_aligned_start,human_aligned_end,bact_aligned_start,bact_aligned_end,evalue_foldseek,griid_gene,iuis_iei,iuis_diseases,bacterial_length,bacterial_domains,in_conserved_module,module_count,module_partner_genes,plddt_human_mean,plddt_bacteria_mean,tm_pvalue_human,tm_pvalue_bacteria,tm_pvalue,q_value,structure_source_human,structure_source_bacteria,alignment_status,immune_relevant,composite_score,alphafold_model_version,run_date,human_gene_in_IUIS
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,f64,bool,bool,str,i64,str,bool,i64,str,f64,f64,f64,f64,f64,f64,str,str,str,bool,f64,str,str,bool
"""Q47ZY9""","""A0A2E5CU08""","""167879""","""Q8IZH9""","""unreviewed""","""GTP cyclohydrolase 1 (EC 3.5.4…",null,"""177""",null,null,"""regulation of blood pressure […","""cytoplasm [GO:0005737]; nucleu…","""GTP binding [GO:0005525]; GTP …","""cytoplasm [GO:0005737]; nucleu…","""GO:0003934; GO:0005525; GO:000…","""1.10.286.10:FF:000003;3.30.113…","""cd00642;""","""1.10.286.10;3.30.1130.10;""","""IPR043133;IPR043134;IPR001474;…","""PTHR11109:SF11;PTHR11109;""","""PF01227;""","""PS00859;PS00860;""",null,"""NP_001019195.1;""","""Dpd""","""Dpd""","""Colwellia psychrerythraea""","""GCF_000012325_NC_003910_Dpd_1""","""GCF_000012325.1_NC_003910_0264…","""GCF_000012325.1_NC_003910_0265…",1,202,"""DOMAIN 2..176; /note=""GTP cycl…","""Dpd__FolE""",0.991,0.964,0.51,0.28,986,175,1,176,6,181,0.0,false,false,null,182,"""DOMAIN 3..180; /note=""GTP cycl…",false,0,null,95.53,97.12,8.8818e-16,2.5535e-15,2.5535e-15,1.7883e-9,"""afdb_v6""","""afdb_v6""","""ok""",false,0.9962,"""model_v6""","""""",null
"""A0A5B9E1W6""","""A0A7C2NS94""","""582702""","""Q5QP19""","""unreviewed""","""Cysteine desulfurase (EC 2.8.1…","""NFS1 hCG_38220""","""397""","""NFS1""",null,"""[2Fe-2S] cluster assembly [GO:…","""cytosol [GO:0005829]; mitochon…","""cysteine desulfurase activity …","""cytosol [GO:0005829]; mitochon…","""GO:0005654; GO:0005829; GO:000…","""3.40.640.10:FF:000003;3.90.115…",null,"""3.90.1150.10;3.40.640.10;""","""IPR000192;IPR020578;IPR010240;…","""PTHR11601:SF34;PTHR11601;""","""PF00266;""","""PS00595;""",null,null,"""Dnd""","""Dnd_ABCDE""","""Rhodanobacter glycinis""","""GCF_008000795_NZ_CP042807_Dnd_…","""GCF_008000795.1_NZ_CP042807_02…","""GCF_008000795.1_NZ_CP042807_02…",1,202,"""DOMAIN 2..361; /note=""Aminotra…","""Dnd__DndA""",0.9237,0.9496,1.3,0.54,1874,367,1,376,9,386,0.0,false,false,null,386,"""DOMAIN 7..369; /note=""Aminotra…",false,0,null,94.52,95.11,1.3656e-14,4.6629e-15,1.3656e-14,1.7883e-9,"""afdb_v6""","""afdb_v6""","""ok""",false,0.9863,"""model_v6""","""""",false
"""Q47ZY9""","""A0A2E5CU08""","""167879""","""Q8IZH8""","""unreviewed""","""GTP cyclohydrolase 1 (EC 3.5.4…",null,"""158""",null,null,"""regulation of blood pressure […","""cytoplasm [GO:0005737]; nucleu…","""GTP binding [GO:0005525]; GTP …","""cytoplasm [GO:0005737]; nucleu…","""GO:0003934; GO:0005525; GO:000…","""1.10.286.10:FF:000003;3.30.113…",null,"""1.10.286.10;3.30.1130.10;""","""IPR043133;IPR043134;IPR001474;…","""PTHR11109:SF11;PTHR11109;""","""PF01227;""","""PS00859;PS00860;""",null,null,"""Dpd""","""Dpd""","""Colwellia psychrerythraea""","""GCF_000012325_NC_003910_Dpd_1""","""GCF_000012325.1_NC_003910_0264…"

In [18]:
n = df_comp_ann["human_gene_in_IUIS"].sum()
n

387